# Lab 3 · QLoRA Fine-Tuning ⭐

**~55 minutes**, of which ~20 is the model training while you drink coffee.

This is the centre of the workshop. Everything the deck said about
adapters, low rank, LoRA and quantization becomes a number you can read
off your own screen.

> ↳ Slides: *Full Fine Tuning* · *What is an Adapter* · *What is Low Rank*
> · *LoRA* · *What is Quantization* · *QLoRA* · *VRAM: training vs inference*

In [ ]:
# --- Install the fine-tuning stack --------------------------------------
# Kaggle ships a matched torch/CUDA pair. --no-deps stops pip replacing torch
# with an incompatible build, which shows up later as baffling CUDA errors.
#
# Takes 2-4 minutes. "dependency conflict" warnings here are expected and fine.
# Output is deliberately NOT suppressed: if this step fails, you need to see it.
!pip install -q --no-deps unsloth unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets huggingface_hub sentencepiece protobuf

print("\ninstall finished - verifying imports...")
import importlib
missing = [m for m in ("unsloth", "trl", "peft", "bitsandbytes", "datasets")
           if importlib.util.find_spec(m) is None]
print("MISSING: " + ", ".join(missing) if missing else "all packages importable")

In [ ]:
# --- Locate the workshop repo -------------------------------------------
# Tries, in order: already present -> attached Kaggle Dataset -> git clone.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Ankush610/LLM-lab.git"

def find_repo() -> Path:
    for candidate in [Path("/kaggle/working/LLM-lab"), Path.cwd(), Path.cwd().parent]:
        if (candidate / "common" / "config.py").exists():
            return candidate
    for d in Path("/kaggle/input").glob("*"):          # attached as a Dataset
        if (d / "common" / "config.py").exists():
            return d
    print("Repo not found locally, cloning...")        # last resort
    subprocess.run(["git", "clone", "-q", REPO_URL, "/kaggle/working/LLM-lab"], check=True)
    return Path("/kaggle/working/LLM-lab")

REPO = find_repo()
sys.path[:0] = [str(REPO / "common"), str(REPO / "dataset")]
print(f"repo: {REPO}")

import config
print(config.summary())

In [ ]:
# --- Hugging Face authentication ----------------------------------------
# Reads the Kaggle Secret named HF_TOKEN. Never paste a token into a cell:
# it is saved with the notebook and shared notebooks leak tokens constantly.
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("Hugging Face: authenticated")
except Exception as e:
    print(f"Hugging Face: NOT authenticated ({type(e).__name__})")
    print("  Fix: right panel -> Add-ons -> Secrets -> add HF_TOKEN, tick the box.")
    print("  Or set USE_UNGATED_MODEL = True below to skip the gated model entirely.")

## 4.1 Why not just fine-tune everything?

Before touching LoRA, let's price up full fine-tuning of this 3B model.

Training needs far more than the weights. For each trainable parameter
Adam keeps two extra state tensors (momentum and variance), plus a
gradient, all typically in fp32.

In [ ]:
P = 3.21e9          # Llama-3.2-3B parameter count
GB = 2**30

full = {
    "weights (fp16)":          P * 2,
    "gradients (fp16)":        P * 2,
    "Adam momentum (fp32)":    P * 4,
    "Adam variance (fp32)":    P * 4,
}

print("  FULL FINE-TUNING")
for k, v in full.items():
    print(f"    {k:<26} {v/GB:>7.1f} GB")
print(f"    {'-'*26} {'-'*10}")
print(f"    {'subtotal':<26} {sum(full.values())/GB:>7.1f} GB   (+ activations)")
print(f"\n    A free T4 has 15 GB. This does not fit. Not close.")

r, n_target = 16, 7
lora_params = P * 0.007     # measured below - about 0.7%
qlora = {
    "weights (4-bit, frozen)": P * 0.5,
    "LoRA weights (fp16)":     lora_params * 2,
    "LoRA gradients (fp16)":   lora_params * 2,
    "Adam state (8-bit)":      lora_params * 2,
}
print("\n  QLoRA")
for k, v in qlora.items():
    print(f"    {k:<26} {v/GB:>7.2f} GB")
print(f"    {'-'*26} {'-'*10}")
print(f"    {'subtotal':<26} {sum(qlora.values())/GB:>7.2f} GB   (+ activations)")
print("\n    Fits, with room for a batch. That is the entire pitch.")

## Load the model

Same 4-bit load as Lab 1. The base weights stay **frozen** for the whole
of training — we never compute a gradient for them.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = config.MODEL_NAME,
    max_seq_length = config.MAX_SEQ_LENGTH,
    dtype          = None,               # None = auto-detect from the GPU
    load_in_4bit   = config.LOAD_IN_4BIT,
)
print(f"\nloaded {config.MODEL_NAME}")
print(f"dtype in use: {model.dtype}")

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template=config.CHAT_TEMPLATE)
print(f"chat template: {config.CHAT_TEMPLATE}")

## 4.2 Attaching the adapter

The idea in one line: instead of updating a big weight matrix `W`
(dimensions d×d), freeze it and learn a small correction

```
ΔW = B @ A        where A is r×d and B is d×r
```

With `r = 16` and `d = 3072`, `W` has 9.4M parameters while `A` and `B`
together have 98k — about 1%. At inference you can either add `ΔW` into
`W` (merging, Lab 5) or keep it separate and swap adapters at will.

Every argument below:

| argument | meaning |
|---|---|
| `r` | the rank — the bottleneck dimension. Capacity knob. |
| `lora_alpha` | scaling; the update is multiplied by `alpha/r`. Convention `alpha = 2r`. |
| `target_modules` | which matrices get an adapter |
| `lora_dropout` | regularisation; 0 is fastest in Unsloth's fused kernels |
| `use_gradient_checkpointing` | recompute activations instead of storing them — slower, much less memory |

> ↳ Slides: *What is an Adapter* · *What is Low Rank* · *LoRA* · *3 Weight Matrix Projections: K Q V*

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = config.LORA_R,
    lora_alpha                 = config.LORA_ALPHA,
    lora_dropout               = config.LORA_DROPOUT,
    bias                       = config.LORA_BIAS,
    target_modules             = config.TARGET_MODULES,
    use_gradient_checkpointing = "unsloth",
    random_state               = config.SEED,
    use_rslora                 = False,
    loftq_config               = None,
)
print("\nadapter attached to:", ", ".join(config.TARGET_MODULES))

## 4.3 The payoff number

This is the slide that made LoRA famous, as a number from your own run.

We count with `count_params` from Lab 1 rather than summing `numel()`:
the frozen base is 4-bit and packs two values per byte, so a naive total
comes out at 1.80 B and the trainable share looks twice as large as it
is. The adapter itself is fp16, so that half of the ratio is unaffected.

In [ ]:
from paramcount import count_params

total, trainable = count_params(model)

print(f"  trainable parameters   {trainable:>15,}")
print(f"  total parameters       {total:>15,}")
print(f"  trainable share        {trainable/total:>15.3%}")
print()
print(f"  Adapter on disk (fp16) {trainable*2/2**20:>12.0f} MB")
print(f"  Full model on disk     {total*2/2**30:>12.1f} GB")
print()
print(f"  We are training {trainable/total:.1%} of the network and leaving")
print(f"  the other {1-trainable/total:.1%} exactly as Meta shipped it.")

## 4.4 Pair exercise — what does `r` actually buy? (5 min)

Work with the person next to you. Before running the cell, predict: if
`r` goes from 16 to 64, what happens to the trainable parameter count?

Then run it.

In [ ]:
# Arithmetic only - we are not rebuilding the model, just counting.
d_model, n_layers = model.config.hidden_size, model.config.num_hidden_layers
n_targets = len(config.TARGET_MODULES)

print(f"  hidden size {d_model}, {n_layers} layers, "
      f"{n_targets} adapted matrices per layer\n")
print(f"  {'r':>4} {'trainable params':>18} {'adapter MB':>12} {'vs r=16':>9}")
print("  " + "-" * 48)

base = None
for r in (4, 8, 16, 32, 64, 128):
    approx = n_layers * n_targets * (2 * d_model * r)
    if r == 16:
        base = approx
    print(f"  {r:>4} {approx:>18,} {approx*2/2**20:>11.0f}M "
          f"{approx/base:>8.2f}x")

print("\n  Linear in r. Doubling the rank doubles the adapter.")
print("  More capacity is not automatically better: high r on a small")
print("  dataset overfits, and you pay for it in both memory and time.")

## 4.5 The dataset and the trainer

Reload the ChatML data from Lab 2 and format it with the same template.

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={
    "train": str(REPO / "dataset/formats/chatml_train.jsonl"),
    "val":   str(REPO / "dataset/formats/chatml_val.jsonl"),
})

def formatting(batch):
    return {"text": [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in batch["messages"]
    ]}

ds = ds.map(formatting, batched=True, remove_columns=["messages"])
print(ds)

### Training arguments

**Batch size and gradient accumulation are one knob split in two.**
`per_device_train_batch_size=2` with `gradient_accumulation_steps=4` gives
an effective batch of 8: the GPU processes 2 at a time and accumulates
gradients over 4 micro-batches before stepping. Same maths as a batch of
8, a quarter of the activation memory, roughly 4× the wall-clock.

`learning_rate=2e-4` looks enormous next to pretraining rates (~1e-5).
It's normal for LoRA: we're training a small, freshly-initialised adapter,
not nudging a converged network.

> ↳ Slide: *Training with SFT-Trainer*

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir                  = str(config.WORK_DIR / "training_output"),
    per_device_train_batch_size = config.BATCH_SIZE,
    gradient_accumulation_steps = config.GRAD_ACCUM,
    warmup_steps                = config.WARMUP_STEPS,
    num_train_epochs            = config.NUM_EPOCHS,
    max_steps                   = config.MAX_STEPS or -1,
    learning_rate               = config.LEARNING_RATE,
    logging_steps               = config.LOGGING_STEPS,
    optim                       = config.OPTIMIZER,
    weight_decay                = config.WEIGHT_DECAY,
    lr_scheduler_type           = config.LR_SCHEDULER,
    seed                        = config.SEED,
    report_to                   = "none",
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    dataset_text_field          = "text",
    max_seq_length              = config.MAX_SEQ_LENGTH,
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = ds["train"],
    args          = args,
)

steps = len(ds["train"]) * config.NUM_EPOCHS // (config.BATCH_SIZE * config.GRAD_ACCUM)
print(f"  effective batch size  {config.BATCH_SIZE * config.GRAD_ACCUM}")
print(f"  optimizer steps       ~{steps}")
print(f"  precision             {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")

### Train on responses only

From Lab 2: mask the prompt so loss is computed on answers only.

In [ ]:
from unsloth.chat_templates import train_on_responses_only

if "llama" in config.CHAT_TEMPLATE:
    inst, resp = "<|start_header_id|>user<|end_header_id|>", \
                 "<|start_header_id|>assistant<|end_header_id|>"
else:
    inst, resp = "<|im_start|>user\n", "<|im_start|>assistant\n"

trainer = train_on_responses_only(trainer,
                                  instruction_part=inst,
                                  response_part=resp)
print(f"loss computed on completions only (marker: {resp!r})")

## 4.7 VRAM before training

Note this number. We compare against it while training runs — it's the
deck's claim that training costs 4–5× inference, measured on your GPU.

> ↳ Slide: *VRAM Usage on Training vs Inference*

In [ ]:
import torch
torch.cuda.reset_peak_memory_stats()
before = torch.cuda.memory_allocated() / 2**30
print(f"  model loaded, not yet training:  {before:.2f} GB")
print(f"  total VRAM on this GPU:          "
      f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB")

## 4.6 Train

**~15–20 minutes on a T4.** Start your break now.

Watch the loss column. What you want:

- a **falling** loss, fast at first then flattening
- **not** `0.0` — that means the label mask is eating everything
- **not** flat — that means the learning rate is too low or nothing is trainable
- **not** `nan` — fp16 overflow; lower the LR

In [ ]:
stats = trainer.train()

### While that ran — training vs inference memory

Run this now the training is done to see the peak.

In [ ]:
peak = torch.cuda.max_memory_allocated() / 2**30
print(f"  inference only (before training)  {before:>6.2f} GB")
print(f"  peak during training              {peak:>6.2f} GB")
print(f"  ratio                             {peak/before:>6.2f}x\n")
print("  The extra is gradients, optimizer state and stored activations.")
print("  The deck said 4-5x for FULL fine-tuning. Ours is lower because")
print("  QLoRA only keeps optimizer state for 0.7% of the parameters -")
print("  which is the whole reason this fits on a free GPU.")

m = stats.metrics
print(f"\n  runtime      {m['train_runtime']/60:.1f} min")
print(f"  final loss   {m['train_loss']:.4f}")

### Reading the loss curve

In [ ]:
history = [h for h in trainer.state.log_history if "loss" in h]
if history:
    losses = [h["loss"] for h in history]
    lo, hi = min(losses), max(losses)
    print(f"  step   loss")
    for h in history[::max(1, len(history)//18)]:
        width = int((h["loss"] - lo) / (hi - lo + 1e-9) * 42)
        print(f"  {h['step']:>4}   {h['loss']:.4f}  {'#' * width}")
    print(f"\n  first {losses[0]:.4f}  ->  last {losses[-1]:.4f}   "
          f"({(losses[0]-losses[-1])/losses[0]:+.1%})")
    if losses[-1] < 0.01:
        print("\n  WARNING: loss near zero usually means the label mask is wrong.")

## Save the adapter

Only the LoRA weights — around 90 MB, not 6 GB. The base model is
unchanged and can be re-downloaded any time.

In [ ]:
model.save_pretrained(str(config.ADAPTER_DIR))
tokenizer.save_pretrained(str(config.ADAPTER_DIR))

import subprocess
size = subprocess.run(["du", "-sh", str(config.ADAPTER_DIR)],
                      capture_output=True, text=True).stdout.split()[0]
print(f"\nadapter saved to {config.ADAPTER_DIR}  ({size})")
print("\nfiles:")
for f in sorted(config.ADAPTER_DIR.iterdir()):
    print(f"  {f.name:<34} {f.stat().st_size/2**20:>8.1f} MB")

## Done

You have trained a language model. The adapter is on disk.

Next notebook asks the same twelve questions from Lab 1 and puts the
answers side by side.

---

### Next: `04_before_after.ipynb` — the payoff

> **Kaggle tip:** if the session has been idle a while, check the right-hand
> panel still shows the GPU attached before starting the next notebook.